# Final model training for recent data 

## Bitcoin model training

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib

from feature_engineering import calculate_features

# -----------------------------
# STEP 1: Load FULL Bitcoin data
# -----------------------------
btc_df = pd.read_csv("preprocessed_datasets\BTC_preprocessed.csv")   # <-- your raw file (not preprocessed)
btc_df["Date"] = pd.to_datetime(btc_df["Date"])
btc_df = btc_df.sort_values("Date").reset_index(drop=True)

# If your feature function expects Date as index
btc_df = btc_df.set_index("Date")

# -----------------------------
# STEP 2: Apply SAME feature engineering
# -----------------------------
btc_feat = calculate_features(btc_df.copy())

# IMPORTANT: drop NaNs created by rolling/lags
btc_feat = btc_feat.dropna()

# -----------------------------
# STEP 3: Define features + target
# -----------------------------
TARGET_COL = "Close"
SEQ_LEN = 30

FEATURE_COLS = [col for col in btc_feat.columns if col != TARGET_COL]

X_all = btc_feat[FEATURE_COLS].values.astype(np.float32)
y_all = btc_feat[[TARGET_COL]].values.astype(np.float32)

print("Feature shape:", X_all.shape)
print("Target shape:", y_all.shape)
print("Num features:", len(FEATURE_COLS))

Feature shape: (916, 29)
Target shape: (916, 1)
Num features: 29


In [6]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib

# -----------------------------
# STEP 2: Define features + target
# -----------------------------
TARGET_COL = "Close"
SEQ_LEN = 30

# Use the exact same feature columns you used before
FEATURE_COLS = [col for col in btc_feat.columns if col != TARGET_COL]

X_all = btc_feat[FEATURE_COLS].values.astype(np.float32)
y_all = btc_feat[[TARGET_COL]].values.astype(np.float32)

print("X shape:", X_all.shape)
print("y shape:", y_all.shape)
print("Num features:", len(FEATURE_COLS))

# -----------------------------
# STEP 3: Fit scalers on full data
# -----------------------------
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_all_scaled = x_scaler.fit_transform(X_all)
y_all_scaled = y_scaler.fit_transform(y_all)

# Save scalers for dashboard
joblib.dump(x_scaler, "btc_final_x_scaler.pkl")
joblib.dump(y_scaler, "btc_final_y_scaler.pkl")

print("Scaled X shape:", X_all_scaled.shape)
print("Scaled y shape:", y_all_scaled.shape)

# -----------------------------
# STEP 4: Create sequences
# -----------------------------
def create_sequences(X, y, seq_len=30):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)

btc_X_seq, btc_y_seq = create_sequences(X_all_scaled, y_all_scaled, SEQ_LEN)

print("Sequence X shape:", btc_X_seq.shape)   # (samples, 30, features)
print("Sequence y shape:", btc_y_seq.shape)   # (samples, 1)

X shape: (916, 29)
y shape: (916, 1)
Num features: 29
Scaled X shape: (916, 29)
Scaled y shape: (916, 1)
Sequence X shape: (886, 30, 29)
Sequence y shape: (886, 1)


In [10]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# -----------------------------
# STEP 5: Define BiLSTM model
# -----------------------------
class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=78, num_layers=2, output_size=1):
        super(BiLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)

        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]
        out = self.fc(out)
        return out

# -----------------------------
# STEP 6: Prepare training
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_size = btc_X_seq.shape[2]

model_btc = BiLSTM(
    input_size=input_size,
    hidden_size=78,   # Bitcoin best hidden size
    num_layers=2,     # Bitcoin best num layers
    output_size=1
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_btc.parameters(), lr=0.0007451964123638622)  

batch_size = 32
epochs = 100   # Bitcoin final best was Adam, 100 epochs

dataset = TensorDataset(
    torch.tensor(btc_X_seq, dtype=torch.float32),
    torch.tensor(btc_y_seq, dtype=torch.float32)
)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

# -----------------------------
# STEP 7: Train model
# -----------------------------
model_btc.train()
for epoch in range(epochs):
    epoch_loss = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        outputs = model_btc(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss / len(loader):.6f}")

# -----------------------------
# STEP 8: Save final BTC model + scalers
# -----------------------------
torch.save(model_btc.state_dict(), "Bitcoin_Final.pth")
joblib.dump(x_scaler, "btc_final_x_scaler.pkl")
joblib.dump(y_scaler, "btc_final_y_scaler.pkl")

print("Saved files:")
print("Bitcoin_Final.pth")
print("btc_final_x_scaler.pkl")
print("btc_final_y_scaler.pkl")

Epoch [10/100] Loss: 0.001298
Epoch [20/100] Loss: 0.007244
Epoch [30/100] Loss: 0.001540
Epoch [40/100] Loss: 0.001494
Epoch [50/100] Loss: 0.001495
Epoch [60/100] Loss: 0.001674
Epoch [70/100] Loss: 0.001690
Epoch [80/100] Loss: 0.001680
Epoch [90/100] Loss: 0.001499
Epoch [100/100] Loss: 0.001669
Saved files:
Bitcoin_Final.pth
btc_final_x_scaler.pkl
btc_final_y_scaler.pkl


In [11]:
model_btc.eval()

with torch.no_grad():
    # take last sequence from training data
    last_seq = btc_X_seq[-1]   # shape (seq_len, features)

    # convert to batch format
    last_seq = torch.tensor(last_seq, dtype=torch.float32).unsqueeze(0).to(device)

    pred_scaled = model_btc(last_seq).cpu().numpy()

    # inverse scale
    pred_price = y_scaler.inverse_transform(pred_scaled)[0, 0]

print("Predicted next close:", pred_price)

Predicted next close: 39626.984


In [12]:
print("Last actual close:", btc_feat["Close"].iloc[-1])

Last actual close: 44167.33203125


## Binance model training

In [9]:
import pandas as pd
import numpy as np
from feature_engineering import calculate_features

# -----------------------------
# STEP 1: Load FULL Binance data
# -----------------------------
bnb_df = pd.read_csv("preprocessed_datasets\BNB_preprocessed.csv")   # change if your file name is different
bnb_df["Date"] = pd.to_datetime(bnb_df["Date"])
bnb_df = bnb_df.sort_values("Date").reset_index(drop=True)

# If your feature function expects Date as index
bnb_df = bnb_df.set_index("Date")

# -----------------------------
# STEP 2: Apply SAME feature engineering
# -----------------------------
bnb_feat = calculate_features(bnb_df.copy())

# Drop NaNs created by rolling/lags
bnb_feat = bnb_feat.dropna()

print("Feature-engineered Binance shape:", bnb_feat.shape)
print(bnb_feat.head())

Feature-engineered Binance shape: (916, 30)
                 Close        High         Low        Open        Volume  \
Date                                                                       
2021-06-30  303.295868  304.801361  281.778015  300.958801  1.903538e+09   
2021-07-01  288.218414  303.527374  281.579132  303.527374  1.357795e+09   
2021-07-02  287.423096  290.621674  277.350311  287.754456  1.133633e+09   
2021-07-03  298.237122  302.605865  283.434021  287.215607  1.113777e+09   
2021-07-04  307.732086  314.713013  292.787384  298.113556  1.387396e+09   

            Daily_Return  Close_Lag_1  Close_Lag_7  Close_Lag_14  \
Date                                                               
2021-06-30      0.010274   300.211548   294.490295    347.033447   
2021-07-01     -0.049712   303.295868   308.397034    352.738525   
2021-07-02     -0.002759   288.218414   281.620911    336.809692   
2021-07-03      0.037624   287.423096   279.438049    335.712830   
2021-07-04     

In [10]:
import joblib
from sklearn.preprocessing import MinMaxScaler

# -----------------------------
# STEP 3: Define features + target
# -----------------------------
TARGET_COL = "Close"
SEQ_LEN = 30

FEATURE_COLS = [col for col in bnb_feat.columns if col != TARGET_COL]

X_all = bnb_feat[FEATURE_COLS].values.astype(np.float32)
y_all = bnb_feat[[TARGET_COL]].values.astype(np.float32)

print("X shape:", X_all.shape)
print("y shape:", y_all.shape)
print("Num features:", len(FEATURE_COLS))

# -----------------------------
# STEP 4: Fit scalers on full data
# -----------------------------
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_all_scaled = x_scaler.fit_transform(X_all)
y_all_scaled = y_scaler.fit_transform(y_all)

# Save scalers with your final names
joblib.dump(x_scaler, "bnb_final_x_scaler.pkl")
joblib.dump(y_scaler, "bnb_final_y_scaler.pkl")

print("Scaled X shape:", X_all_scaled.shape)
print("Scaled y shape:", y_all_scaled.shape)

# -----------------------------
# STEP 5: Create sequences
# -----------------------------
def create_sequences(X, y, seq_len=30):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)

bnb_X_seq, bnb_y_seq = create_sequences(X_all_scaled, y_all_scaled, SEQ_LEN)

print("Sequence X shape:", bnb_X_seq.shape)
print("Sequence y shape:", bnb_y_seq.shape)

X shape: (916, 29)
y shape: (916, 1)
Num features: 29
Scaled X shape: (916, 29)
Scaled y shape: (916, 1)
Sequence X shape: (886, 30, 29)
Sequence y shape: (886, 1)


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# -----------------------------
# STEP 6: Define BiLSTM model
# -----------------------------
class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=107, num_layers=1, output_size=1):
        super(BiLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_size).to(x.device)

        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]
        out = self.fc(out)
        return out

# -----------------------------
# STEP 7: Prepare training
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_size = bnb_X_seq.shape[2]

model = BiLSTM(
    input_size=input_size,
    hidden_size=107,   # Binance best hidden size
    num_layers=1,      # Binance best num layers
    output_size=1
).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.0013949192871148132)

batch_size = 32
epochs = 50   

dataset = TensorDataset(
    torch.tensor(bnb_X_seq, dtype=torch.float32),
    torch.tensor(bnb_y_seq, dtype=torch.float32)
)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

# -----------------------------
# STEP 8: Train model
# -----------------------------
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss / len(loader):.6f}")

# -----------------------------
# STEP 9: Save final Binance model + scalers
# -----------------------------
torch.save(model.state_dict(), "Binance_Final.pth")

print("Saved files:")
print("Binance_Final.pth")
print("bnb_final_x_scaler.pkl")
print("bnb_final_y_scaler.pkl")

Epoch [10/50] Loss: 0.001469
Epoch [20/50] Loss: 0.001789
Epoch [30/50] Loss: 0.002713
Epoch [40/50] Loss: 0.004079
Epoch [50/50] Loss: 0.000932
Saved files:
Binance_Final.pth
bnb_final_x_scaler.pkl
bnb_final_y_scaler.pkl


In [12]:
model.eval()

with torch.no_grad():
    last_seq = bnb_X_seq[-1]   # last sequence

    last_seq = torch.tensor(last_seq, dtype=torch.float32).unsqueeze(0).to(device)

    pred_scaled = model(last_seq).cpu().numpy()

    pred_price = y_scaler.inverse_transform(pred_scaled)[0, 0]

print("Predicted next close:", pred_price)

Predicted next close: 307.26852


In [13]:
print("Last actual close:", bnb_feat["Close"].iloc[-1])

Last actual close: 314.4082946777344
